# Notebook 03 — Training

Trains one configuration and records it. Everything that varies between runs is set in `config.py`, so this notebook is the same for every experiment and the run is fully described by the configuration it read.

Each run writes its own folder under `experiments/<run_id>/` — settings, checkpoints and history — and one line into `results/experiments.csv`.

Runs on Colab: `albumentations` provides the online augmentation and cannot be installed on the local Windows machine.

# 0. Environment Setup

Mounts Drive, makes `src/` importable and seeds the random number generators.

Check the reported device before going further: on CPU a single epoch takes minutes rather than seconds, and the comparison table requires every run to share the same hardware — batch size is bound by GPU memory and affects the result.

In [ ]:
# =====================================================
# 0.0 - SESSION SETUP: WORK FROM THE SESSION DISK
# =====================================================
# Drive is 35x slower per file than /content, measured, and the Dataset opens
# five files per sample. Training from Drive turns a 38-second epoch into
# roughly three minutes, so the project is copied once per session.
#
# Drive stays the canonical copy. src/ is refreshed from it every time, which
# is cheap and removes the trap of training a module you updated on Drive but
# never re-copied. Results live on /content until cell 6.1 writes them back,
# and /content does not survive the session.
import shutil
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/VineProject_ToF")
LOCAL_ROOT = Path("/content/VineProject_ToF")

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

if (LOCAL_ROOT / "src" / "config.py").exists():
    shutil.copytree(DRIVE_ROOT / "src", LOCAL_ROOT / "src", dirs_exist_ok=True)
    print(f"{LOCAL_ROOT} already there, src/ refreshed from Drive")
else:
    print(f"Copying the project to {LOCAL_ROOT}, takes a couple of minutes ...")
    shutil.copytree(DRIVE_ROOT, LOCAL_ROOT, dirs_exist_ok=True)
    print("copied")

In [ ]:
# =====================================================
# 0.1 - BOOTSTRAP: DRIVE MOUNT AND MODULE PATH
# =====================================================
import importlib
import sys
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/VineProject_ToF")
LOCAL_ROOT = Path("/content/VineProject_ToF")

if Path("/content").exists():
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    # Reading from Drive goes through FUSE and is slow on small files, and the
    # Dataset opens five per sample. A copy on the session disk is used when
    # one is there; without it nothing changes and Drive is read directly.
    PROJECT_ROOT = LOCAL_ROOT if (LOCAL_ROOT / "src" / "config.py").exists() else DRIVE_ROOT
    print(f"Project root: {PROJECT_ROOT}"
          f"{'  (session disk)' if PROJECT_ROOT == LOCAL_ROOT else '  (Drive)'}")
else:
    PROJECT_ROOT = Path.cwd().parent

if not (PROJECT_ROOT / "src" / "config.py").exists():
    raise FileNotFoundError(f"config.py not found under: {PROJECT_ROOT / 'src'}")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

import config


def reload_project_modules() -> None:
    """Reload the modules in src/ after editing them.

    Python caches imported modules, so a module edited on Drive is not picked
    up by a simple re-import. Run this cell after uploading a new version,
    then re-run the cells that import from it. Order matters: each module
    reads the constants of the previous one at import time.
    """
    for module_name in (
        "config", "data_preparation", "exploration",
        "dataset", "losses", "metrics", "models", "training",
    ):
        if module_name in sys.modules:
            importlib.reload(sys.modules[module_name])
            print(f"  reloaded: {module_name}")


config.print_environment_info()
config.set_seed()

In [ ]:
# =====================================================
# 0.2 - CONFIGURE THIS RUN
# =====================================================
# The only cell that changes between runs. It must stay AFTER the bootstrap:
# cell 0.1 reloads every module, which would discard anything set before it.
#
# Nothing here edits config.py, so no file has to be copied again and no
# restart is needed. build_config() deep-copies at call time, so the run's
# config.json and its row in experiments.csv record what was actually used
# rather than the defaults in the file.
config.ARCHITECTURE = "UNet"                 # UNet | SegNet | PSPNet
config.LOSS_CONFIG["type"] = "ce"            # ce | dice | combined | focal
config.SKELETON_LOSS_CONFIG["enabled"] = False

RUN_SUFFIX = "base"                          # what tells sibling runs apart

# --- variations, each commented out until the stage that needs it ----------
# config.RANDOM_SEED = 123; config.set_seed()          # noise floor
# config.OPTIMIZER_CONFIG["type"] = "sgd"              # optimiser probe
# config.OPTIMIZER_CONFIG["lr"] = 1e-2                 # SGD wants a higher lr
# config.SCHEDULER_CONFIG["type"] = "cosine"           # scheduler probe
# config.set_enabled_modalities(confidence=False)      # channel ablation
# config.CLASS_WEIGHT_CONFIG["apply_weights"] = False  # focal probe

In [ ]:
# =====================================================
# 0.3 - WHAT THIS RUN WILL BE
# =====================================================
# Printed before anything is built, so a run is never started by accident with
# the configuration of the previous one still in place.
from data_preparation import get_splits_hash

SPLITS_HASH = get_splits_hash()

print(f"Architecture:  {config.ARCHITECTURE} (base {config.MODEL_BASE_CHANNELS[config.ARCHITECTURE]})")
print(f"Input:         {config.INPUT_CHANNELS} channels, {config.IMG_SIZE[0]}x{config.IMG_SIZE[1]}")
print(f"Loss:          {config.LOSS_CONFIG['type']}, class weights "
      f"{config.CLASS_WEIGHT_CONFIG['method'] if config.CLASS_WEIGHT_CONFIG['apply_weights'] else 'off'}")
print(f"Skeleton loss: {'on, weight ' + str(config.SKELETON_LOSS_CONFIG['weight']) if config.SKELETON_LOSS_CONFIG['enabled'] else 'off'}")
print(f"Optimizer:     {config.OPTIMIZER_CONFIG['type']}, lr {config.OPTIMIZER_CONFIG['lr']}")
print(f"Split:         {SPLITS_HASH}")

# 1. Data

Builds the augmentation pipeline and the three dataloaders.

Normalisation uses the per-channel statistics measured in notebook 02. They carry the fingerprint of the split they were measured on, so values left over from a different partition are reported rather than applied silently.

Augmentation is geometric only. Photometric transforms would change what a depth value and a confidence value mean, and applying them to intensity alone would break the agreement between channels describing the same pixel.

In [ ]:
# =====================================================
# 1.1 - TRANSFORMS AND DATALOADERS
# =====================================================
from dataset import build_transforms, create_dataloaders

TRANSFORMS = build_transforms(splits_hash=SPLITS_HASH)
DATALOADERS = create_dataloaders(
    transforms=TRANSFORMS,
    load_skeleton=config.SKELETON_LOSS_CONFIG["load_skeleton"],
)

In [ ]:
# =====================================================
# 1.2 - WHAT THE MODEL WILL ACTUALLY RECEIVE
# =====================================================
# The online pipeline is random, so this is the only place where its output is
# visible. A flip or rotation applied to the image but not to its label would
# be obvious here and invisible in any metric until the model failed to learn.
from exploration import show_augmented_batch

show_augmented_batch(DATALOADERS["train"], n_samples=3)

# 2. Model

Builds the architecture named in `config.ARCHITECTURE` and checks it before any epoch runs.

The width comes from `config.MODEL_BASE_CHANNELS`, chosen so the three architectures have comparable capacity — otherwise the comparison would confound architecture with size.

`test_model` runs a forward **and a backward** pass: memory peaks during the backward, so a model that forwards fine can still exhaust the GPU once gradients are allocated. On CUDA the peak is reported, which is what tells you whether the batch size fits.

In [ ]:
# =====================================================
# 2.1 - BUILD AND CHECK THE MODEL
# =====================================================
from models import count_parameters, create_model, test_model

MODEL = create_model().to(config.DEVICE)

if not test_model(MODEL):
    raise RuntimeError("Model check failed — do not start training.")

# 3. Loss and Optimizer

Class weights are reloaded from disk, not recomputed: they were measured on the training split alone during EDA, so the loss never sees the data it is evaluated on.

The skeleton recall loss is built only when `config.SKELETON_LOSS_CONFIG['enabled']` is set. Running once with it and once without is the ablation the notes describe in section 7.6 — theory predicts it trades precision for recall, and this dataset already tends to over-segment canes.

In [ ]:
# =====================================================
# 3.1 - LOSS, OPTIMIZER, SCHEDULER
# =====================================================
from losses import create_loss_function, create_skeleton_loss
from training import create_optimizer, create_scheduler

CLASS_WEIGHTS = config.load_class_weights(expected_splits_hash=SPLITS_HASH)
CRITERION = create_loss_function(class_weights=CLASS_WEIGHTS).to(config.DEVICE)
SKELETON_CRITERION = create_skeleton_loss()

OPTIMIZER = create_optimizer(MODEL)
SCHEDULER = create_scheduler(OPTIMIZER)

if CLASS_WEIGHTS is not None:
    for name, weight in zip(config.CLASS_NAMES, CLASS_WEIGHTS.tolist()):
        print(f"  {name:<12} weight {weight:.2f}")

# 4. Training

Runs the loop. Each epoch trains, validates and updates the schedule.

Two artefacts are written on different schedules, because they cost three orders of magnitude apart: `history.json` every epoch, a few kilobytes, so the numbers of an interrupted run survive in readable form; `last.pt` every `config.CHECKPOINT_INTERVAL` epochs and at the end, tens of megabytes, as the point training resumes from. The best checkpoint is written whenever the monitored metric improves, whatever the interval.

A session that drops can be picked up by setting `RUN_ID` to the interrupted run and re-running the cell: the history continues unbroken, so the curves look the same as if it had never stopped.

The per-epoch line reports precision and recall next to the monitored metric. A term that rewards recall without penalising false positives moves the two apart, and the mean IoU alone would hide it.

clDice is computed on validation each epoch. It measures connectivity and is blind to radius inflation by design, so it separates *the structure was found* from *the boundary is imprecise* — a distinction the overlap scores cannot make.

Output: `experiments/<run_id>/`

In [ ]:
# =====================================================
# 4.1 - RUN THE TRAINING
# =====================================================
# A Colab session can drop mid-run. find_resume_point() returns the checkpoint
# of an existing run if there is one, and train_model() continues from it:
# weights, optimiser and scheduler state, history, best value and the epoch
# counter are all restored, so the run proceeds as though it had not stopped.
#
# To resume, set RUN_ID to the identifier of the interrupted run instead of
# generating a new one, then re-run this cell unchanged.
from training import find_resume_point, train_model

RUN_ID = config.build_run_id(RUN_SUFFIX)   # or: RUN_ID = "<interrupted run>"
RESUME_FROM = find_resume_point(RUN_ID)

if RESUME_FROM is not None:
    print(f"Resuming {RUN_ID} from {RESUME_FROM.name}")

TRACKER, BEST_CHECKPOINT = train_model(
    model=MODEL,
    dataloaders=DATALOADERS,
    criterion=CRITERION,
    optimizer=OPTIMIZER,
    scheduler=SCHEDULER,
    skeleton_criterion=SKELETON_CRITERION,
    run_id=RUN_ID,
    resume_from=RESUME_FROM,
)

In [ ]:
# =====================================================
# 4.2 - LEARNING CURVES
# =====================================================
FIGURES_DIR = config.get_run_dir(RUN_ID, create=True)["figures"]

TRACKER.plot_history(
    metrics=("loss", "mean_iou_no_bg", "mean_precision", "mean_recall"),
    save_path=FIGURES_DIR / "history.png",
)

# 5. Test Evaluation

Loads the best checkpoint — not the model as it stands, which is whatever the last epoch left behind — and runs it once over the test split.

This is the only time the test split is touched. Everything up to here was decided on training and validation.

Output: one line in `results/experiments.csv`

In [ ]:
# =====================================================
# 5.1 - EVALUATE THE BEST CHECKPOINT ON TEST
# =====================================================
from metrics import print_cldice_report
from training import load_checkpoint, run_epoch

load_checkpoint(BEST_CHECKPOINT, MODEL, device=config.DEVICE)

TEST_METRICS = run_epoch(
    model=MODEL,
    dataloader=DATALOADERS["test"],
    criterion=CRITERION,
    device=config.DEVICE,
    skeleton_criterion=SKELETON_CRITERION,
    skeleton_weight=config.SKELETON_LOSS_CONFIG["weight"],
    compute_centreline_dice=True,
    description="test",
)

print(f"Test loss: {TEST_METRICS['loss']:.4f}")
print(f"Mean IoU:  {TEST_METRICS['mean_iou_no_bg']:.4f} (no background)")
print()
print_cldice_report({
    "cldice_per_class": TEST_METRICS["cldice_per_class"],
    "precision_per_class": TEST_METRICS["cldice_precision_per_class"],
    "recall_per_class": TEST_METRICS["cldice_recall_per_class"],
    "mean_cldice": TEST_METRICS["mean_cldice"],
})

In [ ]:
# =====================================================
# 5.2 - APPEND THE RUN TO THE RESULTS TABLE
# =====================================================
from training import append_result, build_result_row

ROW = build_result_row(
    run_id=RUN_ID,
    tracker=TRACKER,
    test_metrics=TEST_METRICS,
    splits_hash=SPLITS_HASH,
    parameters=count_parameters(MODEL),
    skeleton_applied=SKELETON_CRITERION is not None,
    notes="",
)

append_result(ROW)

In [ ]:
# =====================================================
# 5.3 - THE TABLE SO FAR
# =====================================================
# Sorted by the monitored metric. The columns shown are the ones that
# distinguish runs; the CSV holds the per-class detail as well.
import pandas as pd

RESULTS = pd.read_csv(config.RESULTS_CSV)

RESULTS[[
    "run_id", "architecture", "parameters", "skeleton_loss_applied",
    "test_mean_iou_no_bg", "test_mean_cldice",
    "test_mean_precision", "test_mean_recall",
    "epochs_run", "git_commit",
]].sort_values("test_mean_iou_no_bg", ascending=False)

# 6. Before The Session Ends

`/content` disappears with the session. Everything this run produced lives there and nowhere else until the next cell copies it back.

Run it after the last run of a session — or after every run, since it costs seconds and a lost session costs an hour.

In [ ]:
# =====================================================
# 6.1 - COPY RESULTS BACK TO DRIVE
# =====================================================
# Only the outputs travel back: the data and the code came from Drive and are
# unchanged. dirs_exist_ok lets this run repeatedly in one session.
import shutil

if PROJECT_ROOT == LOCAL_ROOT:
    for folder in ("experiments", "results"):
        source = PROJECT_ROOT / folder
        if source.exists():
            shutil.copytree(source, DRIVE_ROOT / folder, dirs_exist_ok=True)
            print(f"  {folder} -> {DRIVE_ROOT / folder}")
    print("\nSafe to close the session.")
else:
    print("Working on Drive already, nothing to copy.")